In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import fastf1
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from urllib.parse import quote

import warnings
warnings.filterwarnings("ignore")

In [11]:
# Enable FastF1 caching
import os
os.makedirs("f1_cache", exist_ok=True)
fastf1.Cache.enable_cache("f1_cache")

# import requests, fastf1, logging
#
# print(fastf1.__version__)
# r = requests.get("https://livetiming.formula1.com/static/2025/Index.json", timeout=10)
# print(r.status_code, r.text[:200])
#
# fastf1.set_log_level(logging.DEBUG)  # then rerun session_2025.load() for the full traceback

### 2025 Data

In [12]:
# Load 2025 Italian GP race session, lap and sector times
session_2025 = fastf1.get_session(2025, "Italy", "R")
session_2025.load()

events      WARNING 	Correcting user input 'Italy' to 'Italian Grand Prix'
core           INFO 	Loading data for Italian Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
_api          DEBUG 	Falling back to livetiming mirror (https://livetiming-mirror.fastf1.dev)
logger      WARNING 	Failed to load session info data!
logger        DEBUG 	Traceback for failure in session info data
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/fastf1/logger.py", line 112, in __wrapped
    return func(*args, **kwargs)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/fastf1/core.py", line 1452, in _load_session_info
    self._session_info = api.session_info(self.api_path,
                         ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
                                          livedata=livedata)
 

In [ ]:
# laps_2025 = session_2025.laps[["Driver", "LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]].copy()
# laps_2025.dropna(inplace=True)

In [14]:
BASE = "https://raw.githubusercontent.com/TracingInsights/2025/main"
EVENT, SESSION = "Italian Grand Prix", "Race"
DRIVERS = ["ALB", "ALO", "ANT", "BEA", "BOR", "COL", "GAS", "HAD", "HAM", "HUL",
           "LAW", "LEC", "NOR", "OCO", "PIA", "RUS", "SAI", "STR", "TSU", "VER"]

frames = []
for drv in DRIVERS:
    r = requests.get(f"{BASE}/{quote(EVENT)}/{quote(SESSION)}/{drv}/laptimes.json", timeout=15)
    if r.status_code != 200:
        continue
    d = r.json()
    frames.append(pd.DataFrame({
        "Driver": drv, "LapNumber": d["lap"], "LapTime": d["time"],
        "Sector1Time": d["s1"], "Sector2Time": d["s2"], "Sector3Time": d["s3"],
    }))

laps_2025 = pd.concat(frames, ignore_index=True)
time_cols = ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]
laps_2025[time_cols] = (laps_2025[time_cols]
                        .apply(pd.to_numeric, errors="coerce")
                        .apply(pd.to_timedelta, unit="s"))
laps_2025.dropna(inplace=True)
laps_2025

,Driver,LapNumber,LapTime,Sector1Time,Sector2Time,Sector3Time
1,ALB,2,0 days 00:01:25.762000,0 days 00:00:27.809000,0 days 00:00:29.380000,0 days 00:00:28.573000
2,ALB,3,0 days 00:01:25.238000,0 days 00:00:27.410000,0 days 00:00:29.424000,0 days 00:00:28.404000
3,ALB,4,0 days 00:01:25.427000,0 days 00:00:27.608000,0 days 00:00:29.388000,0 days 00:00:28.431000
4,ALB,5,0 days 00:01:24.598000,0 days 00:00:27.170000,0 days 00:00:29.072000,0 days 00:00:28.356000
5,ALB,6,0 days 00:01:24.743000,0 days 00:00:27.261000,0 days 00:00:29.151000,0 days 00:00:28.331000
...,...,...,...,...,...,...
970,VER,49,0 days 00:01:21.328000,0 days 00:00:26.925000,0 days 00:00:27.674000,0 days 00:00:26.729000
971,VER,50,0 days 00:01:21.297000,0 days 00:00:26.934000,0 days 00:00:27.544000,0 days 00:00:26.819000
972,VER,51,0 days 00:01:21.749000,0 days 00:00:27.306000,0 days 00:00:27.690000,0 days 00:00:26.753000
973,VER,52,0 days 00:01:21.003000,0 days 00:00:26.927000,0 days 00:00:27.512000,0 days 00:00:26.564000


In [15]:
laps_2025.to_csv('ItalianGP_laps_2025.csv', index=False)

In [18]:
# Convert times to seconds
for col in ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
    laps_2025[f"{col} (s)"] = laps_2025[col].dt.total_seconds()

laps_2025

,Driver,LapNumber,LapTime,Sector1Time,Sector2Time,Sector3Time,LapTime (s),Sector1Time (s),Sector2Time (s),Sector3Time (s)
1,ALB,2,0 days 00:01:25.762000,0 days 00:00:27.809000,0 days 00:00:29.380000,0 days 00:00:28.573000,85.762,27.809,29.380,28.573
2,ALB,3,0 days 00:01:25.238000,0 days 00:00:27.410000,0 days 00:00:29.424000,0 days 00:00:28.404000,85.238,27.410,29.424,28.404
3,ALB,4,0 days 00:01:25.427000,0 days 00:00:27.608000,0 days 00:00:29.388000,0 days 00:00:28.431000,85.427,27.608,29.388,28.431
4,ALB,5,0 days 00:01:24.598000,0 days 00:00:27.170000,0 days 00:00:29.072000,0 days 00:00:28.356000,84.598,27.170,29.072,28.356
5,ALB,6,0 days 00:01:24.743000,0 days 00:00:27.261000,0 days 00:00:29.151000,0 days 00:00:28.331000,84.743,27.261,29.151,28.331
...,...,...,...,...,...,...,...,...,...,...
970,VER,49,0 days 00:01:21.328000,0 days 00:00:26.925000,0 days 00:00:27.674000,0 days 00:00:26.729000,81.328,26.925,27.674,26.729
971,VER,50,0 days 00:01:21.297000,0 days 00:00:26.934000,0 days 00:00:27.544000,0 days 00:00:26.819000,81.297,26.934,27.544,26.819
972,VER,51,0 days 00:01:21.749000,0 days 00:00:27.306000,0 days 00:00:27.690000,0 days 00:00:26.753000,81.749,27.306,27.690,26.753
973,VER,52,0 days 00:01:21.003000,0 days 00:00:26.927000,0 days 00:00:27.512000,0 days 00:00:26.564000,81.003,26.927,27.512,26.564


In [19]:
# Group by driver to get average sector times per driver
sector_times_2025 = laps_2025.groupby("Driver")[["Sector1Time (s)", "Sector2Time (s)", "Sector3Time (s)"]].mean().reset_index()

sector_times_2025

,Driver,Sector1Time (s),Sector2Time (s),Sector3Time (s)
0,ALB,27.615942,28.428365,27.840519
1,ALO,28.422565,28.833609,29.346304
2,ANT,27.678923,28.570788,27.738769
3,BEA,27.808327,28.666769,27.788346
4,BOR,27.806192,28.486462,27.796962
5,COL,28.064235,28.693255,27.898412
6,GAS,27.970000,28.766373,27.861392
7,HAD,27.773962,28.545846,27.782846
8,HAM,27.576981,28.416577,27.684981
9,LAW,27.822346,28.765231,27.895538


### Quali Data 2026

In [ ]:
# Qualifying data for 2026 Italian GP (Monza)
# Time = lap from the last session each driver reached (Q3 for P1-10, Q2 for P11-16, Q1 for P17-22)
qualifying_2026 = pd.DataFrame({
    "Driver": ["GAS", "RUS", "PIA", "LEC", "HAM", "VER", "ANT", "COL", "NOR", "LIN",
               "BOR", "BEA", "HUL", "LAW", "SAI", "OCO",
               "TSU", "ALB", "BOT", "PER", "ALO", "STR"],
    "QualifyingTime (s)": [81.786, 81.846, 81.966, 82.004, 82.011, 82.070, 82.093, 82.220, 82.256, 82.286,
                           82.517, 82.756, 82.779, 82.821, 83.453, 83.454,
                           83.755, 84.356, 84.364, 84.595, 85.150, 85.222]
})

### Weather component

In [ ]:
API_KEY = "put your API here"
# Monza (Autodromo Nazionale Monza), Italy -- 2026 Italian GP venue
weather_url = f"http://api.openweathermap.org/data/2.5/forecast?lat=45.6156&lon=9.2811&appid={API_KEY}&units=metric"
response = requests.get(weather_url)
weather_data = response.json()
print(weather_data)

In [ ]:
# Extract the relevant weather data for the race (Sunday at 3pm local time)
forecast_time = "2026-09-06 15:00:00"
forecast_data = None
if "list" in weather_data:
    for forecast in weather_data["list"]:
        if forecast["dt_txt"] == forecast_time:
            forecast_data = forecast
            break
else:
    print(f"Weather API call failed ({weather_data.get('message', weather_data)}); using default values.")

if forecast_data:
    rain_probability = forecast_data["pop"]
    temperature = forecast_data["main"]["temp"]
else:
    rain_probability = 0
    temperature = 20

In [ ]:
# Merge qualifying data with sector times data
merged_data = qualifying_2026.merge(sector_times_2025, left_on="Driver", right_on="Driver", how="left")

In [ ]:
# Create weather features for the model
merged_data["RainProbability"] = rain_probability
merged_data["Temperature"] = temperature

In [ ]:
# Attach average lap time to merged_data the same way sector times were attached (left join),
# so a driver with no clean lap (e.g. retired, like LEC) is dropped consistently from X and y.
merged_data["LapTime (s)"] = merged_data["Driver"].map(laps_2025.groupby("Driver")["LapTime (s)"].mean())
merged_data.dropna(subset=["LapTime (s)"], inplace=True)

# Define feature set (Qualifying + Sector Times + Weather)
X = merged_data[["QualifyingTime (s)", "Sector1Time (s)", "Sector2Time (s)", "Sector3Time (s)", "RainProbability", "Temperature"]].fillna(0)

# Use average lap time as the target
y = merged_data["LapTime (s)"]

## Train Gradient Boosting Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=38)
model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=38)
model.fit(X_train, y_train)

In [ ]:
# Predict race times using 2026 qualifying and sector data
predicted_race_times = model.predict(X)
merged_data["PredictedRaceTime (s)"] = predicted_race_times
race_predictions_2026 = merged_data.sort_values(by="PredictedRaceTime (s)")

In [ ]:
# Evaluate
y_pred = model.predict(X_test)
print(f"\n Model Error (MAE): {mean_absolute_error(y_test, y_pred):.2f} seconds")

## Random Forest Regressor

In [ ]:
# Tune Random Forest hyperparameters with 5-fold grid search
rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}
rf_grid_search = GridSearchCV(
    RandomForestRegressor(random_state=38),
    rf_param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
rf_grid_search.fit(X_train, y_train)
rf = rf_grid_search.best_estimator_
print(f"Best RF params: {rf_grid_search.best_params_}")

rf_y_pred = rf.predict(X_test)
print(f"\n Model Error (MAE): {mean_absolute_error(y_test, rf_y_pred):.2f} seconds")

# Predict race times for all drivers using the tuned Random Forest model
merged_data["PredictedRaceTime_RF (s)"] = rf.predict(X)

### Predicted vs Actual Result (2026 Italian GP)

In [ ]:
# Actual 2026 Italian GP race result (formula1.com/en/results/2026/races/1293/italy/race-result)
actual_results_2026 = pd.DataFrame({
    "Driver": ["ANT", "RUS", "VER", "NOR", "PIA", "HAM", "GAS", "LIN", "COL", "TSU",
               "BOR", "HUL", "SAI", "LAW", "BEA", "OCO", "ALB", "PER", "BOT",
               "STR", "ALO", "LEC"],
    "ActualPosition": list(range(1, 23)),
    "Status": ["Finished"] * 19 + ["DNF (26 laps)", "DNF (23 laps)", "DNF (1 lap)"]
})

## Positions and Lap times prediction using RF

In [ ]:
# Predicted finishing order (Random Forest ranking): rank drivers by predicted average lap time, fastest first.
# ActualAvgLapTime (s) comes from the real race laps loaded via FastF1 (merged_data),
# not the results page, since lapped cars there only show "+1 lap" / "+2 laps", not a time.
predicted_ranking = merged_data.sort_values("PredictedRaceTime_RF (s)")[
    ["Driver", "LapTime (s)", "PredictedRaceTime_RF (s)"]
].reset_index(drop=True)
predicted_ranking = predicted_ranking.rename(columns={"LapTime (s)": "ActualAvgLapTime (s)"})
predicted_ranking["PredictedPosition"] = predicted_ranking.index + 1
predicted_ranking["TimeDiff (s)"] = predicted_ranking["ActualAvgLapTime (s)"] - predicted_ranking["PredictedRaceTime_RF (s)"]

# Combine predicted and actual results. Left join on the actual results keeps LEC
# in the table even though the model couldn't predict him (no clean lap data to
# train/predict on after his lap-1 DNF) -- he just gets NaN predicted/actual values.
results_comparison = actual_results_2026.merge(predicted_ranking, on="Driver", how="left")
results_comparison["PositionDiff"] = results_comparison["ActualPosition"] - results_comparison["PredictedPosition"]

# Order from P1 down to last classified position
results_comparison = results_comparison[
    ["Driver", "ActualPosition", "PredictedPosition", "PositionDiff",
     "ActualAvgLapTime (s)", "PredictedRaceTime_RF (s)", "TimeDiff (s)", "Status"]
].sort_values("ActualPosition").reset_index(drop=True)
results_comparison

In [ ]:
# Average position error (LEC excluded -- no prediction was made for him)
mean_abs_position_error = results_comparison["PositionDiff"].abs().mean()
print(f"Mean Absolute Position Error: {mean_abs_position_error:.2f} positions")

## Lap Times up close

In [ ]:
# Lap time comparison: actual vs. Random Forest predicted, per driver
lap_time_comparison = merged_data[["Driver", "LapTime (s)", "PredictedRaceTime_RF (s)"]].copy()
lap_time_comparison = lap_time_comparison.rename(columns={"LapTime (s)": "ActualAvgLapTime (s)"})
lap_time_comparison["TimeDiff (s)"] = lap_time_comparison["ActualAvgLapTime (s)"] - lap_time_comparison["PredictedRaceTime_RF (s)"]
lap_time_comparison

## Visualisations

In [ ]:
# Actual vs Predicted (RF) average lap time per driver, ordered by actual finishing position
COLOR_ACTUAL = "#2a78d6"     # categorical slot 1 (blue)
COLOR_PREDICTED = "#008300"  # categorical slot 2 (green)
INK_PRIMARY = "#0b0b0b"
INK_MUTED = "#898781"
GRID_LINE = "#e1e0d9"

plot_data = results_comparison.dropna(subset=["ActualAvgLapTime (s)"]).sort_values("ActualPosition")

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(plot_data))
bar_width = 0.38

ax.bar(x - bar_width / 2, plot_data["ActualAvgLapTime (s)"], bar_width,
       label="Actual", color=COLOR_ACTUAL)
ax.bar(x + bar_width / 2, plot_data["PredictedRaceTime_RF (s)"], bar_width,
       label="Predicted (RF)", color=COLOR_PREDICTED)

ax.set_xticks(x)
ax.set_xticklabels(plot_data["Driver"], color=INK_PRIMARY)
ax.set_ylabel("Average Lap Time (s)", color=INK_PRIMARY)
ax.set_title("Actual vs Predicted Average Lap Time — 2026 Italian GP", color=INK_PRIMARY)
ax.set_ylim(plot_data[["ActualAvgLapTime (s)", "PredictedRaceTime_RF (s)"]].min().min() - 1,
            plot_data[["ActualAvgLapTime (s)", "PredictedRaceTime_RF (s)"]].max().max() + 1)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(GRID_LINE)
ax.spines["bottom"].set_color(GRID_LINE)
ax.tick_params(colors=INK_MUTED)
ax.yaxis.grid(True, color=GRID_LINE, linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, labelcolor=INK_PRIMARY)

plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs Actual finishing position (RF model). Points on the diagonal were
# predicted exactly right; above the line means the model expected a worse finish
# than the driver actually achieved, and vice versa below. Points are colored by
# each driver's team (FastF1's 2026 team colors).
import fastf1.plotting as fp

pos_data = results_comparison.dropna(subset=["PredictedPosition"])
point_colors = [fp.get_driver_color(d, session_2025) for d in pos_data["Driver"]]

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([1, 22], [1, 22], linestyle="--", linewidth=1.5, color=INK_MUTED, label="Perfect prediction")
ax.scatter(pos_data["ActualPosition"], pos_data["PredictedPosition"],
           s=70, color=point_colors, edgecolors="white", linewidths=0.8, zorder=3)

for _, row in pos_data.iterrows():
    ax.annotate(row["Driver"], (row["ActualPosition"], row["PredictedPosition"]),
                textcoords="offset points", xytext=(6, 4), fontsize=8, color=INK_PRIMARY)

ax.set_xlabel("Actual Position", color=INK_PRIMARY)
ax.set_ylabel("Predicted Position (RF)", color=INK_PRIMARY)
ax.set_title("Predicted vs Actual Finishing Position — 2026 Italian GP", color=INK_PRIMARY)
ax.set_xlim(0, 23)
ax.set_ylim(0, 23)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(GRID_LINE)
ax.spines["bottom"].set_color(GRID_LINE)
ax.tick_params(colors=INK_MUTED)
ax.grid(True, color=GRID_LINE, linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, labelcolor=INK_PRIMARY, loc="upper left")

plt.tight_layout()
plt.show()

Random Forest Regressor scored better than Gradient Boost Regressor by 0.09 seconds MAE, making it better model. Prediction on final position varies, but looking at lap times RF predicts well. Potential improvement is tuning models and comparing. Error of 0.19 s is significant if we consider this is Formula 1.

## RF Tuning

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300, 600],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": np.linspace(0.01, 1.0, 10, endpoint=True),
    "min_samples_leaf": np.linspace(0.1, 0.5, 5, endpoint=True),
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=38),
    param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
best_rf_pred = best_rf.predict(X_test)

print(f"Best params: {grid_search.best_params_}")
print(f"Test MAE: {mean_absolute_error(y_test, best_rf_pred):.2f} seconds")